# 03 — Tendances & projection
Projection simple de la **fécondité** et de la **population** à l'horizon 2035,
à partir des tendances réelles observées.

In [1]:

import os, warnings, json, re, unicodedata
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw"); PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo"); FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS): os.makedirs(d, exist_ok=True)

def deacc(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-demographie


In [2]:

nat = pd.read_csv(os.path.join(PROC, "dhs_national_wide.csv"))
wb = pd.read_csv(os.path.join(PROC, "worldbank_wide.csv"))
from sklearn.linear_model import LinearRegression


### 1. Projection de la fécondité (régression linéaire sur les EDS récentes)

In [3]:

t = nat.dropna(subset=["FE_FRTR_W_TFR"])
t = t[t["annee"] >= 1997]                      # tendance récente
X = t[["annee"]].values; y = t["FE_FRTR_W_TFR"].values
lr = LinearRegression().fit(X, y)
fut = np.arange(t["annee"].min(), 2036).reshape(-1,1)
pred = lr.predict(fut)
# année estimée d'atteinte du seuil de renouvellement (2,1)
an_seuil = (2.1 - lr.intercept_) / lr.coef_[0]
fig, ax = plt.subplots()
ax.plot(t["annee"], y, "o", color="#1f4e79", label="EDS observé")
ax.plot(fut.ravel(), pred, "--", color="#c0392b", label="Tendance projetée")
ax.axhline(2.1, color="grey", ls=":"); ax.text(1998, 2.2, "Renouvellement (2,1)", fontsize=8, color="grey")
ax.legend(); ax.set_ylabel("Indice de fécondité"); ax.set_title("Projection de la fécondité au Sénégal")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "08_projection_fecondite.png"), bbox_inches="tight")
plt.close(fig)
print("Fécondité projetée 2035 : %.1f enfants/femme" % lr.predict([[2035]])[0])
print("Seuil de renouvellement (2,1) atteint vers :", int(round(an_seuil)))


Fécondité projetée 2035 : 3.5 enfants/femme
Seuil de renouvellement (2,1) atteint vers : 2058


### 2. Projection de la population (taux de croissance récent)

In [4]:

p = wb.dropna(subset=["SP.POP.TOTL"]).sort_values("annee")
pop0 = p["SP.POP.TOTL"].iloc[-1]; an0 = int(p["annee"].iloc[-1])
g = wb.dropna(subset=["SP.POP.GROW"])["SP.POP.GROW"].iloc[-5:].mean() / 100
years = list(range(an0, 2036))
proj = [pop0 * (1+g)**(yr-an0) for yr in years]
fig, ax = plt.subplots()
ax.plot(p["annee"], p["SP.POP.TOTL"]/1e6, color="#1f4e79", lw=2, label="Observé (BM)")
ax.plot(years, np.array(proj)/1e6, "--", color="#c0392b", lw=2, label=f"Projection (+{g*100:.1f}%/an)")
ax.legend(); ax.set_ylabel("Population (millions)"); ax.set_title("Projection de la population du Sénégal (2035)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "09_projection_population.png"), bbox_inches="tight")
plt.close(fig)
pd.DataFrame({"annee": years, "population_projetee": proj}).to_csv(
    os.path.join(MODELS, "projection_population.csv"), index=False, encoding="utf-8-sig")
print("Population %d : %.1f M -> projection 2035 : %.1f M" % (an0, pop0/1e6, proj[-1]/1e6))


Population 2024 : 18.5 M -> projection 2035 : 24.2 M


### Conclusion
- Si la tendance se poursuit, la fécondité continue de **baisser** mais le seuil de
  renouvellement (2,1) ne serait atteint qu'**au-delà de 2035** → la population
  continuera de croître fortement (**dividende démographique** potentiel).
- ⚠️ Projections linéaires simples, à titre illustratif : la réalité dépendra des
  politiques (éducation, planification familiale, santé) et du contexte socio-économique.
